# NER script demo

The same sentence is passed to every NER inference script. Each command prints its result as JSON.

In [3]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working/src/ner').is_dir() else Path(r'D:/quantum_task')
os.chdir(PROJECT_ROOT)
ENV_PATHS = [PROJECT_ROOT / '.env', PROJECT_ROOT / 'src/.env', PROJECT_ROOT / 'src/ner/.env']
load_dotenv()
SENTENCE = 'At sunrise, the climbers began their ascent of Mount Everest.'
BERT_CHECKPOINT = PROJECT_ROOT / 'src/ner/data/results/bert_ce_training/kaggle/working/bert_base_ce_results/checkpoint-1330'
MOUNTAIN_CATALOG = PROJECT_ROOT / 'src/ner/data/catalogs/Mountain.csv'
OPEN_PEAKS_CATALOG = PROJECT_ROOT / 'src/ner/data/catalogs/open_peaks_names.csv'

def run_script(*args):
    completed = subprocess.run(args, cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
    print(completed.stdout.strip())

print(f'sentence={SENTENCE}')
print(f'bert_checkpoint={BERT_CHECKPOINT}')
print(f'env_path={PROJECT_ROOT / "src/.env"}')
print(f'groq_api_configured={bool(os.getenv("GROQ_API"))}')

sentence=At sunrise, the climbers began their ascent of Mount Everest.
bert_checkpoint=D:\quantum_task\src\ner\data\results\bert_ce_training\kaggle\working\bert_base_ce_results\checkpoint-1330
env_path=D:\quantum_task\src\.env
groq_api_configured=True


## GLiNER

GLiNER predicts mountain spans directly from the sentence.

In [2]:
import importlib.util

if importlib.util.find_spec('gliner'):
    run_script('python', 'src/ner/inference/gliner_run.py', SENTENCE, '--threshold', '0.50', '--device', 'auto')
else:
    print('GLiNER demo skipped: install dependencies with pip install -r src/ner/requirements.txt.')

{"model": "urchade/gliner_small-v2.1", "threshold": 0.5, "entities": [{"start": 47, "end": 60, "text": "Mount Everest", "label": "mountain", "score": 0.9880010485649109}]}


## Fine-tuned BERT

The validation-selected CE checkpoint returns BIO-based mountain entities.

In [3]:
run_script('python', 'src/ner/inference/bert_run.py', SENTENCE, '--checkpoint', str(BERT_CHECKPOINT), '--threshold', '0.50', '--device', 'auto')

{"checkpoint": "D:\\quantum_task\\src\\ner\\data\\results\\bert_ce_training\\kaggle\\working\\bert_base_ce_results\\checkpoint-1330", "threshold": 0.5, "entities": [{"text": "Mount Everest", "label": "Mountain", "score": 0.999695897102356, "start": 47, "end": 60}]}


## Catalog matcher

The deterministic matcher maps the mention to a canonical mountain name from the combined catalogs.

In [4]:
run_script('python', 'src/ner/inference/matcher_run.py', SENTENCE, '--catalog', str(MOUNTAIN_CATALOG), '--catalog', str(OPEN_PEAKS_CATALOG), '--fuzzy-threshold', '0.90')

{"catalogs": ["D:\\quantum_task\\src\\ner\\data\\catalogs\\Mountain.csv", "D:\\quantum_task\\src\\ner\\data\\catalogs\\open_peaks_names.csv"], "matches": [{"file": "sentence", "mention": "Mount Everest", "canonical_name": "Mount Everest", "start": 47, "end": 60, "score": 1.0, "match_type": "exact_normalized"}]}


## Groq LLM (optional)

The LLM call loads `GROQ_API_KEY` from `.env` and runs only when the key is configured.

In [5]:
if os.getenv('GROQ_API'):
    run_script('python', 'src/ner/evaluation/llm_benchmark.py', '--text', SENTENCE, '--workers', '1', '--model-name', 'qwen/qwen3.6-27b')
else:
    print('LLM demo skipped: set GROQ_API_KEY to run it.')

Processing batch (1 sentences) with qwen/qwen3.6-27b via Groq SDK...

Text: 'At sunrise, the climbers began their ascent of Mount Everest.'
Mountain: Mount Everest
Request time: 1.18 seconds
----------------------------------------
Total processing time: 1.20 seconds
